In [1]:
# import dependencies
import pandas as pd
import numpy as np
from functools import reduce

## Load ACS

And extract ID and town name

In [2]:
# load ACS files
df_median_income = pd.read_csv(
    "../data/raw/acs/median_household_income/ACSDT5Y2024.B19013-Data.csv", skiprows=1
)
df_percent_elderly = pd.read_csv(
    "../data/raw/acs/sex_by_age/ACSDT5Y2024.B01001-Data.csv", skiprows=1
)
df_median_year_house_built = pd.read_csv(
    "../data/raw/acs/median_year_structure_built/ACSDT5Y2024.B25035-Data.csv",
    skiprows=1,
)
df_vehicle = pd.read_csv(
    "../data/raw/acs/household_vehicles/ACSDT5Y2024.B08201-Data.csv", skiprows=1
)
df_renter_occupied = pd.read_csv(
    "../data/raw/acs/renter_occupied/ACSDT5Y2024.B25003-Data.csv", skiprows=1
)
df_education_level = pd.read_csv(
    "../data/raw/acs/education_level/ACSDT5Y2024.B15003-Data.csv", skiprows=1
)
df_poverty_level = pd.read_csv(
    "../data/raw/acs/poverty_level/ACSDT5Y2024.B17001-Data.csv", skiprows=1
)
df_disability_status = pd.read_csv(
    "../data/raw/acs/disability_status/ACSDT5Y2024.B18101-Data.csv", skiprows=1
)
df_mobile_home = pd.read_csv(
    "../data/raw/acs/mobile_home/ACSDT5Y2024.B25024-Data.csv", skiprows=1
)

In [3]:
# list of all ACS dfs for easier processing
acs_dfs = [
    df_median_income,
    df_percent_elderly,
    df_median_year_house_built,
    df_vehicle,
    df_renter_occupied,
    df_education_level,
    df_poverty_level,
    df_disability_status,
    df_mobile_home,
]

In [4]:
# extract FIPS GEOID from last 10 digits of Geography column
# extract town name (everything before the first comma)
for df in acs_dfs:
    df["GEOID"] = df["Geography"].str[-10:]
    df["town_name"] = df["Geographic Area Name"].str.split(",", n=1).str[0]

## Process individual dataframes

Retain margin of error and some raw counts for future sensitivity analysis or eda. If calculated here, margin of error follows official ACS guidance from the Census Bureau.

In [5]:
# helper function to calculate margin of error (MOE)
def ratio_moe(numerator, numerator_moe, denominator, denominator_moe):
    """
    Calculate the margin of error (MOE) for a ratio or percent, using the ACS-recommended formula.

    MOE = 100 * sqrt((numerator_moe / denominator)^2 + ((numerator * denominator_moe) / denominator^2)^2)

    Based on U.S. Census Bureau ACS guidance. Assumes independence between numerator and denominator.
    Annoyingly complicated. Had to look it up. Statistics hurt my brain.
    """
    return 100 * np.sqrt(
        (numerator_moe / denominator) ** 2
        + ((numerator * denominator_moe) / (denominator**2)) ** 2
    )

In [6]:
# get median income df with only relevant columns
df_median_income = df_median_income[
    [
        "GEOID",
        "town_name",
        "Estimate!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)",
        "Margin of Error!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)",
    ]
].rename(
    columns={
        "Estimate!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)": "median_income",
        "Margin of Error!!Median household income in the past 12 months (in 2024 inflation-adjusted dollars)": "median_income_moe",
    }
)
df_median_income.head()

,GEOID,town_name,median_income,median_income_moe
0,5000100325,Addison town,106667,7181
1,5000108575,Bridport town,83214,17776
2,5000109025,Bristol town,74620,8110
3,5000116000,Cornwall town,114583,32017
4,5000126300,Ferrisburgh town,106989,21771


In [7]:
# calculate percent elderly (65+) by summing relevant columns and dividing by total population


# helper to sum columns by prefix and age group
def sum_age_groups(df, prefix, age_groups):
    cols = [f"{prefix}{age}" for age in age_groups]
    return df[cols].sum(axis=1)


# define age groups for 65+
age_groups = [
    "65 and 66 years",
    "67 to 69 years",
    "70 to 74 years",
    "75 to 79 years",
    "80 to 84 years",
    "85 years and over",
]

# build full column names for estimates and MOEs
# intentionally hardcoding columns, to fail if ACS changes their column names in the future
male_est = [f"Estimate!!Total:!!Male:!!{age}" for age in age_groups]
female_est = [f"Estimate!!Total:!!Female:!!{age}" for age in age_groups]
male_moe = [f"Margin of Error!!Total:!!Male:!!{age}" for age in age_groups]
female_moe = [f"Margin of Error!!Total:!!Female:!!{age}" for age in age_groups]

elderly_cols = male_est + female_est
elderly_moe_cols = male_moe + female_moe

# rename total population columns for easier reference
df_percent_elderly = df_percent_elderly.rename(
    columns={
        "Estimate!!Total:": "total_population",
        "Margin of Error!!Total:": "total_population_moe",
    }
)

# calculate elderly raw counts and MOEs
df_percent_elderly["elderly_65plus"] = df_percent_elderly[elderly_cols].sum(axis=1)
# sum MOEs in quadrature (square root of sum of squares) for elderly population
df_percent_elderly["elderly_65plus_moe"] = np.sqrt(
    (df_percent_elderly[elderly_moe_cols] ** 2).sum(axis=1)
)

# calculate percent and MOE
df_percent_elderly["percent_elderly"] = (
    df_percent_elderly["elderly_65plus"] / df_percent_elderly["total_population"] * 100
)
# MOE_percent = 100 * sqrt( (elderly_moe/total)^2 + (elderly*total_population_moe/total^2)^2 )
df_percent_elderly["percent_elderly_moe"] = ratio_moe(
    df_percent_elderly["elderly_65plus"],
    df_percent_elderly["elderly_65plus_moe"],
    df_percent_elderly["total_population"],
    df_percent_elderly["total_population_moe"],
)

# keep only relevant columns, some renamed for clarity
df_percent_elderly = df_percent_elderly[
    [
        "GEOID",
        "town_name",
        "percent_elderly",
        "percent_elderly_moe",
        "elderly_65plus",
        "elderly_65plus_moe",
        "total_population",
        "total_population_moe",
    ]
]
df_percent_elderly.head()

,GEOID,town_name,percent_elderly,percent_elderly_moe,elderly_65plus,elderly_65plus_moe,total_population,total_population_moe
0,5000100325,Addison town,27.148936,6.322324,319,54.735729,1175,185
1,5000108575,Bridport town,24.291498,6.468954,300,62.665780,1235,204
2,5000109025,Bristol town,21.942827,4.833447,829,182.518492,3778,26
3,5000116000,Cornwall town,25.662252,6.107244,310,51.195703,1208,207
4,5000126300,Ferrisburgh town,23.299511,3.794391,620,100.891030,2661,17


In [8]:
# get median year house built, with relevant columns
df_median_year_house_built = df_median_year_house_built[
    [
        "GEOID",
        "town_name",
        "Estimate!!Median year structure built",
        "Margin of Error!!Median year structure built",
    ]
].rename(
    columns={
        "Estimate!!Median year structure built": "median_year_house_built",
        "Margin of Error!!Median year structure built": "median_year_house_built_moe",
    }
)
df_median_year_house_built.head()

,GEOID,town_name,median_year_house_built,median_year_house_built_moe
0,5000100325,Addison town,1980,6
1,5000108575,Bridport town,1978,6
2,5000109025,Bristol town,1976,5
3,5000116000,Cornwall town,1976,7
4,5000126300,Ferrisburgh town,1968,6


In [9]:
# extract vehicle availability (percent of households with no vehicle available)

# rename columns
df_vehicle = df_vehicle.rename(
    columns={
        "Estimate!!Total:!!No vehicle available": "no_vehicle_available",
        "Margin of Error!!Total:!!No vehicle available": "no_vehicle_available_moe",
        "Estimate!!Total:": "occupied_housing_units",
        "Margin of Error!!Total:": "occupied_housing_units_moe",
    }
)

# calculate percent of households with no vehicle available
df_vehicle["pct_no_vehicle"] = (
    df_vehicle["no_vehicle_available"] / df_vehicle["occupied_housing_units"]
) * 100


# calculate MOE for percent no vehicle using ratio MOE formula
# MOE_percent = 100 * sqrt( (no_vehicle_moe/total)^2 + (no_vehicle*total_moe/total^2)^2 )
df_vehicle["pct_no_vehicle_moe"] = ratio_moe(
    df_vehicle["no_vehicle_available"],
    df_vehicle["no_vehicle_available_moe"],
    df_vehicle["occupied_housing_units"],
    df_vehicle["occupied_housing_units_moe"],
)

# keep only relevant columns
df_vehicle = df_vehicle[
    [
        "GEOID",
        "town_name",
        "pct_no_vehicle",
        "pct_no_vehicle_moe",
        "no_vehicle_available",
        "no_vehicle_available_moe",
        "occupied_housing_units",
        "occupied_housing_units_moe",
    ]
]
df_vehicle.head()

,GEOID,town_name,pct_no_vehicle,pct_no_vehicle_moe,no_vehicle_available,no_vehicle_available_moe,occupied_housing_units,occupied_housing_units_moe
0,5000100325,Addison town,0.612245,0.819537,3,4,490,58
1,5000108575,Bridport town,1.509434,1.344323,8,7,530,88
2,5000109025,Bristol town,6.125739,3.896236,114,72,1861,140
3,5000116000,Cornwall town,1.016260,1.638063,5,8,492,96
4,5000126300,Ferrisburgh town,2.949309,3.326007,32,36,1085,85


In [10]:
# percent of occupied housing units that are renter-occupied

# rename columns
df_renter_occupied = df_renter_occupied.rename(
    columns={
        "Estimate!!Total:!!Renter occupied": "renter_occupied",
        "Margin of Error!!Total:!!Renter occupied": "renter_occupied_moe",
        "Estimate!!Total:": "occupied_housing_units",
        "Margin of Error!!Total:": "occupied_housing_units_moe",
    }
)

# calculate percent renter-occupied
df_renter_occupied["pct_renter_occupied"] = (
    df_renter_occupied["renter_occupied"]
    / df_renter_occupied["occupied_housing_units"]
    * 100
)

# calculate MOE for percent renter-occupied using ratio MOE formula
# MOE_percent = 100 * sqrt( (renter_occupied_moe/occupied_housing_units)^2 + (renter_occupied*occupied_housing_units_moe/occupied_housing_units^2)^2 )
df_renter_occupied["pct_renter_occupied_moe"] = ratio_moe(
    df_renter_occupied["renter_occupied"],
    df_renter_occupied["renter_occupied_moe"],
    df_renter_occupied["occupied_housing_units"],
    df_renter_occupied["occupied_housing_units_moe"],
)

# keep only relevant columns
df_renter_occupied = df_renter_occupied[
    [
        "GEOID",
        "town_name",
        "pct_renter_occupied",
        "pct_renter_occupied_moe",
        "renter_occupied",
        "renter_occupied_moe",
        "occupied_housing_units",
        "occupied_housing_units_moe",
    ]
]
df_renter_occupied.head()

,GEOID,town_name,pct_renter_occupied,pct_renter_occupied_moe,renter_occupied,renter_occupied_moe,occupied_housing_units,occupied_housing_units_moe
0,5000100325,Addison town,10.816327,5.260227,53,25,490,58
1,5000108575,Bridport town,17.924528,9.354012,95,47,530,88
2,5000109025,Bristol town,28.694250,8.033232,534,144,1861,140
3,5000116000,Cornwall town,5.691057,2.866134,28,13,492,96
4,5000126300,Ferrisburgh town,14.654378,6.734520,159,72,1085,85


In [11]:
# calculate percent with bachelor's degree or higher

# define degree categories for bachelor's or higher
degree_categories = [
    "Bachelor's degree",
    "Master's degree",
    "Professional school degree",
    "Doctorate degree",
]

# build full column names for estimates and MOEs
degree_est_cols = [f"Estimate!!Total:!!{deg}" for deg in degree_categories]
degree_moe_cols = [f"Margin of Error!!Total:!!{deg}" for deg in degree_categories]

# rename total population columns for easier reference
df_education_level = df_education_level.rename(
    columns={
        "Estimate!!Total:": "total_population_25+",
        "Margin of Error!!Total:": "total_population_25+_moe",
    }
)

# calculate raw count and MOE for bachelor's or higher
df_education_level["bachelors_or_higher"] = df_education_level[degree_est_cols].sum(
    axis=1
)
df_education_level["bachelors_or_higher_moe"] = np.sqrt(
    (df_education_level[degree_moe_cols] ** 2).sum(axis=1)
)

# calculate percent and MOE
df_education_level["pct_bachelors_or_higher"] = (
    df_education_level["bachelors_or_higher"]
    / df_education_level["total_population_25+"]
    * 100
)
df_education_level["pct_bachelors_or_higher_moe"] = ratio_moe(
    df_education_level["bachelors_or_higher"],
    df_education_level["bachelors_or_higher_moe"],
    df_education_level["total_population_25+"],
    df_education_level["total_population_25+_moe"],
)

# keep only relevant columns
df_education_level = df_education_level[
    [
        "GEOID",
        "town_name",
        "pct_bachelors_or_higher",
        "pct_bachelors_or_higher_moe",
        "bachelors_or_higher",
        "bachelors_or_higher_moe",
        "total_population_25+",
        "total_population_25+_moe",
    ]
]
df_education_level.head()

,GEOID,town_name,pct_bachelors_or_higher,pct_bachelors_or_higher_moe,bachelors_or_higher,bachelors_or_higher_moe,total_population_25+,total_population_25+_moe
0,5000100325,Addison town,40.201568,8.576477,359,60.753601,893,116
1,5000108575,Bridport town,34.825328,9.700377,319,76.249590,916,131
2,5000109025,Bristol town,41.880065,7.705231,1292,227.055059,3085,168
3,5000116000,Cornwall town,54.731183,13.222747,509,80.405224,930,170
4,5000126300,Ferrisburgh town,51.106640,8.320225,1016,150.329638,1988,135


In [12]:
# get percent below the poverty level

# rename columns for easier reference
df_poverty_level = df_poverty_level.rename(
    columns={
        "Estimate!!Total:": "poverty_universe_population",
        "Margin of Error!!Total:": "poverty_universe_population_moe",
        "Estimate!!Total:!!Income in the past 12 months below poverty level:": "below_poverty_total",
        "Margin of Error!!Total:!!Income in the past 12 months below poverty level:": "below_poverty_total_moe",
    }
)

# calculate percent below poverty line
df_poverty_level["pct_below_poverty"] = (
    df_poverty_level["below_poverty_total"]
    / df_poverty_level["poverty_universe_population"]
    * 100
)

# calculate MOE for percent below poverty using ACS ratio formula
df_poverty_level["pct_below_poverty_moe"] = ratio_moe(
    df_poverty_level["below_poverty_total"],
    df_poverty_level["below_poverty_total_moe"],
    df_poverty_level["poverty_universe_population"],
    df_poverty_level["poverty_universe_population_moe"],
)

# keep only relevant columns
df_poverty_level = df_poverty_level[
    [
        "GEOID",
        "town_name",
        "pct_below_poverty",
        "pct_below_poverty_moe",
        "below_poverty_total",
        "below_poverty_total_moe",
        "poverty_universe_population",
        "poverty_universe_population_moe",
    ]
]

df_poverty_level.head()

,GEOID,town_name,pct_below_poverty,pct_below_poverty_moe,below_poverty_total,below_poverty_total_moe,poverty_universe_population,poverty_universe_population_moe
0,5000100325,Addison town,4.510638,3.394277,53,39,1175,185
1,5000108575,Bridport town,4.418985,3.835924,54,46,1222,204
2,5000109025,Bristol town,10.085152,5.482154,379,206,3758,28
3,5000116000,Cornwall town,0.834028,0.600983,10,7,1199,205
4,5000126300,Ferrisburgh town,2.217212,1.428105,59,38,2661,17


In [13]:
# percent disability status (percent of population with a disability)

# define age groups (as in the ACS columns)
age_groups = [
    "Under 5 years",
    "5 to 17 years",
    "18 to 34 years",
    "35 to 64 years",
    "65 to 74 years",
    "75 years and over",
]

# build full column names for estimates and MOEs (male and female, with a disability)
# intentionally hardcoding columns, to fail if ACS changes their column names in the future
male_est = [f"Estimate!!Total:!!Male:!!{age}:!!With a disability" for age in age_groups]
female_est = [
    f"Estimate!!Total:!!Female:!!{age}:!!With a disability" for age in age_groups
]
male_moe = [
    f"Margin of Error!!Total:!!Male:!!{age}:!!With a disability" for age in age_groups
]
female_moe = [
    f"Margin of Error!!Total:!!Female:!!{age}:!!With a disability" for age in age_groups
]

disability_cols = male_est + female_est
disability_moe_cols = male_moe + female_moe

# rename columns for easier reference
df_disability_status = df_disability_status.rename(
    columns={
        "Estimate!!Total:": "civilian_noninstitutionalized_population",
        "Margin of Error!!Total:": "civilian_noninstitutionalized_population_moe",
    }
)

# calculate raw counts and MOEs for disability
df_disability_status["with_disability"] = df_disability_status[disability_cols].sum(
    axis=1
)
# sum MOEs in quadrature (square root of sum of squares) for elderly population
df_disability_status["with_disability_moe"] = np.sqrt(
    (df_disability_status[disability_moe_cols] ** 2).sum(axis=1)
)

# calculate percent and MOE
df_disability_status["percent_with_disability"] = (
    df_disability_status["with_disability"]
    / df_disability_status["civilian_noninstitutionalized_population"]
    * 100
)
df_disability_status["percent_with_disability_moe"] = ratio_moe(
    df_disability_status["with_disability"],
    df_disability_status["with_disability_moe"],
    df_disability_status["civilian_noninstitutionalized_population"],
    df_disability_status["civilian_noninstitutionalized_population_moe"],
)

# keep only relevant columns
df_disability_status = df_disability_status[
    [
        "GEOID",
        "town_name",
        "percent_with_disability",
        "percent_with_disability_moe",
        "with_disability",
        "with_disability_moe",
        "civilian_noninstitutionalized_population",
        "civilian_noninstitutionalized_population_moe",
    ]
]

df_disability_status.head()

,GEOID,town_name,percent_with_disability,percent_with_disability_moe,with_disability,with_disability_moe,civilian_noninstitutionalized_population,civilian_noninstitutionalized_population_moe
0,5000100325,Addison town,11.063830,3.756213,130,39.102430,1175,185
1,5000108575,Bridport town,13.360324,5.285433,165,59.312731,1235,204
2,5000109025,Bristol town,18.706759,6.606142,703,248.203546,3758,28
3,5000116000,Cornwall town,9.850993,4.211954,119,46.615448,1208,207
4,5000126300,Ferrisburgh town,16.272078,4.470369,433,118.924346,2661,17


In [14]:
# get percent mobile home

# rename columns
df_mobile_home = df_mobile_home.rename(
    columns={
        "Estimate!!Total:": "total_housing_units",
        "Margin of Error!!Total:": "total_housing_units_moe",
        "Estimate!!Total:!!Mobile home": "mobile_homes",
        "Margin of Error!!Total:!!Mobile home": "mobile_homes_moe",
    }
)

# calculate percent of mobile homes in town
df_mobile_home["pct_mobile_home"] = (
    df_mobile_home["mobile_homes"] / df_mobile_home["total_housing_units"]
) * 100

# calculate MOE for percent mobile homes using ratio MOE formula
# MOE_percent = 100 * sqrt( (mobile_home_moe/total)^2 + (mobile_homes*total_moe/total^2)^2 )
df_mobile_home["pct_mobile_home_moe"] = ratio_moe(
    df_mobile_home["mobile_homes"],
    df_mobile_home["mobile_homes_moe"],
    df_mobile_home["total_housing_units"],
    df_mobile_home["total_housing_units_moe"],
)

# keep only relevant columns
df_mobile_home = df_mobile_home[
    [
        "GEOID",
        "town_name",
        "pct_mobile_home",
        "pct_mobile_home_moe",
        "mobile_homes",
        "mobile_homes_moe",
        "total_housing_units",
        "total_housing_units_moe",
    ]
]
df_mobile_home.head()

,GEOID,town_name,pct_mobile_home,pct_mobile_home_moe,mobile_homes,mobile_homes_moe,total_housing_units,total_housing_units_moe
0,5000100325,Addison town,8.993902,4.696541,59,30,656,78
1,5000108575,Bridport town,9.146341,5.803389,60,37,656,98
2,5000109025,Bristol town,8.971668,5.914073,171,112,1906,142
3,5000116000,Cornwall town,7.829181,6.211285,44,34,562,101
4,5000126300,Ferrisburgh town,4.076087,2.866666,60,42,1472,100


## Export individual and summary dataframes to csv

Also check that town names and GEOID's match `town_flood_risk.csv` (created in `etl_gis.ipynb`)

In [15]:
# save individual df's to csv
named_dfs = [
    ("median_income", df_median_income),
    ("percent_elderly", df_percent_elderly),
    ("median_year_house_built", df_median_year_house_built),
    ("vehicle", df_vehicle),
    ("renter_occupied", df_renter_occupied),
    ("education_level", df_education_level),
    ("poverty_level", df_poverty_level),
    ("disability_status", df_disability_status),
    ("mobile_home", df_mobile_home),
]

for name, df in named_dfs:
    df.to_csv(f"../data/cleaned/acs/{name}.csv", index=False)

In [16]:
# create summary df of variables of interest and export to csv

# create skinny dfs for each variable
dfs = [
    df_median_income[["GEOID", "town_name", "median_income", "median_income_moe"]],
    df_percent_elderly[
        [
            "GEOID",
            "town_name",
            "total_population",
            "total_population_moe",
            "percent_elderly",
            "percent_elderly_moe",
        ]
    ],
    df_median_year_house_built[
        ["GEOID", "town_name", "median_year_house_built", "median_year_house_built_moe"]
    ],
    df_vehicle[["GEOID", "town_name", "pct_no_vehicle", "pct_no_vehicle_moe"]],
    df_renter_occupied[
        [
            "GEOID",
            "town_name",
            "occupied_housing_units",
            "occupied_housing_units_moe",
            "pct_renter_occupied",
            "pct_renter_occupied_moe",
        ]
    ],
    df_education_level[
        ["GEOID", "town_name", "pct_bachelors_or_higher", "pct_bachelors_or_higher_moe"]
    ],
    df_poverty_level[
        ["GEOID", "town_name", "pct_below_poverty", "pct_below_poverty_moe"]
    ],
    df_disability_status[
        ["GEOID", "town_name", "percent_with_disability", "percent_with_disability_moe"]
    ],
    df_mobile_home[["GEOID", "town_name", "pct_mobile_home", "pct_mobile_home_moe"]],
]

# merge all on GEOID and town_name
final_df = reduce(
    lambda left, right: pd.merge(left, right, on=["GEOID", "town_name"], how="outer"),
    dfs,
)

# export
final_df.to_csv("../data/cleaned/acs_summary.csv", index=False)

final_df.head()

,GEOID,town_name,median_income,median_income_moe,total_population,total_population_moe,percent_elderly,percent_elderly_moe,median_year_house_built,median_year_house_built_moe,...,pct_renter_occupied,pct_renter_occupied_moe,pct_bachelors_or_higher,pct_bachelors_or_higher_moe,pct_below_poverty,pct_below_poverty_moe,percent_with_disability,percent_with_disability_moe,pct_mobile_home,pct_mobile_home_moe
0,5000100325,Addison town,106667,7181,1175,185,27.148936,6.322324,1980,6,...,10.816327,5.260227,40.201568,8.576477,4.510638,3.394277,11.063830,3.756213,8.993902,4.696541
1,5000108575,Bridport town,83214,17776,1235,204,24.291498,6.468954,1978,6,...,17.924528,9.354012,34.825328,9.700377,4.418985,3.835924,13.360324,5.285433,9.146341,5.803389
2,5000109025,Bristol town,74620,8110,3778,26,21.942827,4.833447,1976,5,...,28.694250,8.033232,41.880065,7.705231,10.085152,5.482154,18.706759,6.606142,8.971668,5.914073
3,5000116000,Cornwall town,114583,32017,1208,207,25.662252,6.107244,1976,7,...,5.691057,2.866134,54.731183,13.222747,0.834028,0.600983,9.850993,4.211954,7.829181,6.211285
4,5000126300,Ferrisburgh town,106989,21771,2661,17,23.299511,3.794391,1968,6,...,14.654378,6.734520,51.106640,8.320225,2.217212,1.428105,16.272078,4.470369,4.076087,2.866666


In [ ]:
# check if acs and town flood risk GEOID and town names match up
df_flood_risk = pd.read_csv("../data/cleaned/town_flood_risk.csv")

# ensure GEOID columns are the same type (string) for matching
final_df["GEOID"] = final_df["GEOID"].astype(str)
df_flood_risk["GEOID"] = df_flood_risk["GEOID"].astype(str)

# check for GEOID mismatches
missing_geoids = final_df[~final_df["GEOID"].isin(df_flood_risk["GEOID"])]
print(f"GEOIDs in final_df not in df_flood_risk: {len(missing_geoids)}")
display(missing_geoids)

# check for town name mismatches (case-insensitive, strip whitespace)
final_names = final_df[["GEOID", "town_name"]].copy()
flood_names = df_flood_risk[["GEOID", "town_name"]].copy()
final_names["town_name_clean"] = final_names["town_name"].str.strip().str.lower()
flood_names["town_name_clean"] = flood_names["town_name"].str.strip().str.lower()

# merge on GEOID, compare names
merged = pd.merge(final_names, flood_names, on="GEOID", how="inner")
name_mismatches = merged[merged["town_name_clean_x"] != merged["town_name_clean_y"]]
print(f"GEOIDs with mismatched town names: {len(name_mismatches)}")
display(name_mismatches[["GEOID", "town_name_x", "town_name_y"]])

GEOIDs in final_df not in df_flood_risk: 0


,GEOID,town_name,median_income,median_income_moe,total_population,total_population_moe,percent_elderly,percent_elderly_moe,median_year_house_built,median_year_house_built_moe,...,pct_renter_occupied,pct_renter_occupied_moe,pct_bachelors_or_higher,pct_bachelors_or_higher_moe,pct_below_poverty,pct_below_poverty_moe,percent_with_disability,percent_with_disability_moe,pct_mobile_home,pct_mobile_home_moe


GEOIDs with mismatched town names: 0


,GEOID,town_name_x,town_name_y
